In [ ]:
import torch
import torch.nn as nn
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

from src.models.shPLRNN import shPLRNN
from src.models.encoder_decoder import ConcatEncoder, ReadoutDecoder
from src.training.teacher_forcing import GeneralizedTeacherForcing as GTF
from src.training.training import train_model
from src.extrapolation import FeatureExtrapolation

from data.dataset import get_dataloader, load_dataset, get_datasets, get_control_params

ModuleNotFoundError: No module named 'src.models.teacher_forcing'

In [ ]:
dataset = load_dataset("../data/lorenz63/lorenz63_dataset_standardized.npz")
bif_data = load_dataset("../data/lorenz63/bifurcation_data.npz")
for d in dataset:
    print(d, dataset[d].shape)
print("---")
for d in bif_data:
    print(d, bif_data[d].shape)

In [ ]:
w1_data = torch.load("../results/2026-05-19/full_run_18-47-16_lorenz63_shPLRNN/shPLRNN_features_1_splitting_True/seed_00/wasserstein_distances.pth")
for d in w1_data:
    print(d, w1_data[d].shape)

In [ ]:
w1_data

In [ ]:
X_train, X_test_id, X_test_ood = get_datasets(dataset)

In [ ]:
train_dataloader, test_id_dataloader, test_ood_dataloader = get_dataloader(dataset, batch_size=32)

In [ ]:
for x in test_id_dataloader:
    print(x[0].shape, x[1].shape)
    break

In [ ]:
train_dataloader, test_id_dataloader, test_ood_dataloader = get_dataloader(dataset, batch_size=512, initial_seq_length=2)

In [ ]:
epochs = 100

# check if mps is available and set device accordingly
if torch.cuda.is_available():
    print("CUDA is available. Using CUDA.")
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("MPS is available. Using MPS.")
    device = torch.device("mps")
else:
    print("MPS and CUDA are not available. Using CPU.")
    device = torch.device("cpu")

encoder = ConcatEncoder(obs_dim=X_train.shape[-1], latent_dim=16)
decoder = ReadoutDecoder(latent_dim=16, obs_dim=X_train.shape[-1])
tf = GTF(forcing_dim=X_train.shape[-1])

feature_splitting = True
num_features = 1

model = shPLRNN(obs_dim=X_train.shape[-1], latent_dim=16, hidden_dim=128,
                encoder=encoder, decoder=decoder, tf=tf,
                num_train_subjects=X_train.shape[0], num_features=num_features,
                feature_splitting=feature_splitting)
# model = torch.compile(model)
optim = torch.optim.Adam(model.parameters(), lr=2e-3)

num_batches = 128

# ask user whether to load or to train model
load_model = input("Load model? (y/n): ")
if load_model.lower() == "y":
    if feature_splitting:
        model.load_state_dict(torch.load("shPLRNN_lorenz63_1_feature_split.pth"))
    else:
        if num_features == 1:
            model.load_state_dict(torch.load("shPLRNN_lorenz63.pth"))
        elif num_features == 2:
            model.load_state_dict(torch.load("shPLRNN_lorenz63_2_features.pth"))
else:
    # for e in range(epochs):
    #     epoch_loss = 0
    #     steps_ahead = 4 + (e // 10)
    #     tf_alpha = 0.25 * 0.97 ** e
    #     train_dataloader.set_seq_length(steps_ahead)
    #     for batch_num, (true_seq, subject_idx) in enumerate(train_dataloader):
    #         if batch_num > num_batches: continue
    #         pred_seq = model.forward(true_seq, model.features[subject_idx], tf_alpha=tf_alpha)
    #         loss = torch.mean((pred_seq - true_seq) ** 2)
    #         optim.zero_grad()
    #         loss.backward()
    #         optim.step()
    #         epoch_loss += loss.item()
    #     print(f"Epoch {e+1}/{epochs}, Loss: {epoch_loss/num_batches:.2e}, Steps ahead: {steps_ahead}, TF alpha: {tf_alpha:.4f}")

    # plt.plot(true_seq[0, :, 0].cpu(), label="True")
    # plt.plot(pred_seq[0, :, 0].detach().cpu(), label="Predicted")
    # # plt.ylim(torch.min(true_seq[0, :, 0].cpu()) - 0.5, torch.max(true_seq[0, :, 0].cpu()) + 0.5)
    # plt.legend()
    # plt.show()
    train_model(
        model=model,
        train_dataloader=train_dataloader,
        test_id_dataloader=test_id_dataloader,
        epochs=epochs,
        num_batches_per_epoch=num_batches,
        optimizer=optim,
        steps_ahead=lambda e: 4 + (e // 10),
        tf_alpha=lambda e: 0.25 * 0.97 ** e,
        force_every=1,
        device=device, #torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        regularizers=None,
        verbose=True,
        save_every=999
    )
    # save model
    feature_str = str(num_features) + "_features" if num_features > 1 else "1_feature"
    split_str = "_split" if feature_splitting else ""
    torch.save(model.state_dict(), f"shPLRNN_lorenz63_{feature_str}{split_str}.pth")


In [ ]:
if model.feature_splitting:
    plt.scatter(range(16), model.features_pos[:, 0].detach().numpy(), c="blue", label="Positional Feature 1")
    plt.twinx()
    plt.scatter(range(16), model.features_dyn[:, 0].detach().numpy(), c="orange", label="Dynamical Feature 1")
else:
    plt.scatter(range(16), model.features[:, 0].detach().numpy(), c="blue", label="Feature 1")
    if model.num_features > 1:
        plt.twinx()
        plt.scatter(range(16), model.features[:, 1].detach().numpy(), c="orange", label="Feature 2")

In [ ]:
feature_values = model.features.detach() if not model.feature_splitting else model.features_dyn.detach()
feature_values_pos = model.features_pos.detach() if model.feature_splitting else None
bif_points, pred_obs = model.bifurcation_diagram_points(
    feature_values=feature_values,
    feature_values_pos=feature_values_pos,
    num_ics=13,
    relevant_dim=0,
    rollout_steps=25_000,
    transient_steps=5_000,
    return_trajs=True
)

In [ ]:
for key in bif_points:
    plt.scatter([key] * len(bif_points[key]["extrema"]), bif_points[key]["extrema"], color="blue", s=10, label="extrema" if key == 0 else "")
    plt.scatter([key] * len(bif_points[key]["fixed_points"]), bif_points[key]["fixed_points"], color="red", s=10, label="Fixed points" if key == 0 else "")
plt.xlabel("Feature value")
plt.ylabel("Observed value")
plt.title("Bifurcation diagram")
plt.legend()
plt.show()

In [ ]:
feature_extrapolator = FeatureExtrapolation(model)
feature_extrapolator.fit(torch.linspace(5, 22.5, 16));

In [ ]:
id_cps = get_control_params(dataset, domain="id")
ood_cps = get_control_params(dataset, domain="ood")
id_cps, ood_cps

In [ ]:
if model.feature_splitting:
    feature_values_dyn_extrap, feature_values_pos_extrap = feature_extrapolator.predict(ood_cps)
    plt.scatter(id_cps, model.features_pos[:, 0].detach().numpy(), c="blue", label="Positional Feature 1")
    plt.scatter(id_cps, model.features_dyn[:, 0].detach().numpy(), c="orange", label="Dynamical Feature 1")
    plt.scatter(ood_cps, feature_values_pos_extrap[:, 0], c="blue", marker="x", label="Extrapolated Positional Feature 1")
    plt.scatter(ood_cps, feature_values_dyn_extrap[:, 0], c="orange", marker="x", label="Extrapolated Dynamical Feature 1")
else:
    feature_values_extrap = feature_extrapolator.predict(ood_cps)
    feature_values_pos_extrap = None
    plt.scatter(id_cps, model.features[:, 0].detach().numpy(), c="blue", label="Feature 1")
    if model.num_features > 1:
        plt.scatter(id_cps, model.features[:, 1].detach().numpy(), c="orange", label="Feature 2")
    plt.scatter(ood_cps, feature_values_extrap[:, 0], c="blue", marker="x", label="Extrapolated Feature 1")
    if model.num_features > 1:
        plt.scatter(ood_cps, feature_values_extrap[:, 1], c="orange", marker="x", label="Extrapolated Feature 2")

plt.xlabel("Subject index / Feature value")
plt.ylabel("Feature value")
plt.title("Feature extrapolation")
plt.legend()
plt.show()

In [ ]:
# bifurcation diagram for extrapolated features
bif_points_extrap, pred_obs_extrap = model.bifurcation_diagram_points(
    feature_values=feature_values_dyn_extrap if model.feature_splitting else feature_values_extrap,
    feature_values_pos=feature_values_pos_extrap if model.feature_splitting else None,
    num_ics=13,
    relevant_dim=0,
    rollout_steps=25_000,
    transient_steps=5_000,
    return_trajs=True
)

In [ ]:
in_domain = torch.linspace(5, 22.5, 16)
extrapolated = torch.linspace(22.5, upper_extrap, num_extrap_points)

for key in bif_points:
    plt.scatter([in_domain[key]] * len(bif_points[key]["extrema"]), bif_points[key]["extrema"], color="blue", s=10, label="extrema" if key == 0 else "")
    plt.scatter([in_domain[key]] * len(bif_points[key]["fixed_points"]), bif_points[key]["fixed_points"], color="red", s=10, label="Fixed points" if key == 0 else "")

for key in bif_points_extrap:
    plt.scatter([extrapolated[key]] * len(bif_points_extrap[key]["extrema"]), bif_points_extrap[key]["extrema"], color="blue", s=10, label="extrema (extrapolated)" if key == 0 else "", marker="x")
    plt.scatter([extrapolated[key]] * len(bif_points_extrap[key]["fixed_points"]), bif_points_extrap[key]["fixed_points"], color="red", s=10, label="Fixed points (extrapolated)" if key == 0 else "", marker="x")
plt.xlabel("Feature value")
plt.ylabel("Observed value")
plt.title("Bifurcation diagram (in-domain and extrapolated features)\nNum features: " + str(model.num_features) + ", Feature splitting: " + str(model.feature_splitting))
plt.legend()
plt.show()